# 天气与特征：本轮生成、可用时点及覆盖

本页读取已归档特征审计与字典。以下方法契约解释不同信息组；是否实际入模以本轮 manifest 的特征清单为准，不把尚未启用的信息组说成已完成实验。

In [1]:
ROOT = __import__('pathlib').Path.cwd()
RUN_DIR = ROOT / 'experiments/runs/R0017'

In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

if "ROOT" not in globals():
    ROOT = Path.cwd()
    while not (ROOT / "scripts/build_features.py").exists():
        if ROOT.parent == ROOT:
            raise RuntimeError("请在 Airline-delay 仓库中运行 Notebook")
        ROOT = ROOT.parent
if "RUN_DIR" not in globals():
    candidates = [p for p in (ROOT / "experiments/runs").glob("R*")
                  if (p / "metrics.csv").exists() and (p / "model_manifest.json").exists()]
    if not candidates:
        raise RuntimeError("请先归档预测实验，或手动设置 RUN_DIR")
    RUN_DIR = max(candidates, key=lambda p: int(p.name[1:]))
TABLES = RUN_DIR / "tables"
FIGURES = RUN_DIR / "figures"
def table(name):
    return pd.read_csv(TABLES / name)
def figure(name):
    display(Image(filename=str(FIGURES / name)))
def receipt(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
display(Markdown(f"**归档运行：{RUN_DIR.name}**。本 Notebook 读取该轮表格与图，不重新拟合模型。"))

**归档运行：R0017**。本 Notebook 读取该轮表格与图，不重新拟合模型。

In [3]:
config = receipt("config.json") if (RUN_DIR / "config.json").exists() else {}
model_manifest = receipt("model_manifest.json")
features = model_manifest.get("features", config.get("features", []))
group_name = model_manifest.get("feature_group", config.get("feature_group", "未登记"))
models = model_manifest.get("models", [])
display(Markdown(f"本轮信息组：**{group_name}**；阶段：**{config.get('stage', '未登记')}**；模型：**{', '.join(models)}**。"))
display(pd.DataFrame([{"development_only": model_manifest.get("development_only"),
                       "parent_run_id": config.get("parent_run_id"),
                       "change": config.get("change"), "include_baselines": model_manifest.get("include_baselines")}]))

本轮信息组：**G4B**；阶段：**P4b**；模型：**training_mean, training_median, carry_forward, g4b_linear**。

,development_only,parent_run_id,change,include_baselines
0,True,R0015,Add latest-state TAF prevailing TEMPO and prob...,True


## 1. 实际特征及生成审计

目标与标签时刻是评价元数据，不是预测变量。只在训练折拟合插补、类别处理和缩放；缺失不能按无延误、晴天或已知零填充。

In [4]:
display(pd.DataFrame({"实际入模字段": features}))
if (RUN_DIR / "feature_audit.json").exists():
    audits = receipt("feature_audit.json")
    for section, values in audits.items():
        display(Markdown(f"**{section}**"))
        display(pd.json_normalize(values) if isinstance(values, dict) else values)
else:
    display(Markdown("本轮未归档 feature_audit.json。"))
dictionary_path = RUN_DIR / "feature_dictionary.csv"
if dictionary_path.exists():
    display(pd.read_csv(dictionary_path))
else:
    display(Markdown("本轮未归档特征字典。"))

,实际入模字段
0,airport_id
1,route
2,downstream_airline
3,aircraft_family
4,downstream_distance
...,...
60,taf_probability_rain_flag
61,taf_probability_snow_flag
62,taf_probability_thunder_flag
63,taf_probability_segment_count


**feature_audit**

,development_only,categorical_features,numeric_features,outcome_rule,feature_group,features,counts.input_connections,counts.outside_prediction_utc_scope,counts.scoped_connections,counts.missing_or_nonfinite_target,counts.negative_target,counts.missing_label_time,counts.prediction_not_before_departure,counts.missing_or_invalid_upstream_delay,counts.eligible_connections
0,True,"[airport_id, route, downstream_airline, aircra...","[downstream_distance, downstream_crs_elapsed_t...",finite target >= 0 and prediction_time < label...,G4B,"[airport_id, route, downstream_airline, aircra...",2821852,560077,2261775,0,0,0,0,0,2261775


**history_audit**

,history_window,include_current_upstream,source_start_utc,current_upstream_rule,event_quality_rule,ignored_future_qa_columns,quality_exclusions_are_sequential,missingness_rule,left_truncation_rule,carry_in_limitation,...,counts.duplicate_flight_id_rows,counts.unique_history_flight_ids,counts.unique_history_tails,counts.same_tail_arrival_tie_rows_retained,queries.rows,queries.missing_tail,queries.known_tail_zero_history,queries.history_window_incomplete,queries.current_upstream_added,queries.events_present
0,"[prediction_time - 24 hours, prediction_time)",False,2026-04-01T11:00:00+00:00,Excluded by default per plan: previously compl...,"unique flight_id, present tail, normal_operati...","[tail_schedule_incomplete, schedule_tie, actua...",True,missing tail: n and mean missing; known tail w...,"Known-left-truncation indicator, not a complet...",Flights with prior-local-month departure dates...,...,0,2395389,6069,12,2261775,0,53090,18121,0,2208685


...

**airport_operations_audit**

,historical_window,scheduled_window,historical_departures,planned_rows,historical_quality,plan_assumption,no_event_mean,source_start_rule,left_truncation_rule,carry_in_limitation,left_window_incomplete_queries
0,"[prediction_time-60min, prediction_time)","[prediction_time, prediction_time+60min)",2409279,3057590,Departure clock/delay consistency and unique s...,Archived BTS scheduled times are treated as kn...,"NA; count is zero, training-only imputation",Per-airport source local-month midnight conver...,Indicator of known source-left truncation; zer...,Prior-local-month flights departing late are a...,0


**weather_audit**

,data_id,manifest,manifest_sha256,rows,cohort_unchanged,metar_available_before_noaa,metar_available_after_noaa,taf_available,noaa_backfill,metar_model_columns,taf_model_columns,metar_correction_policy,taf_policy,noaa_policy,assumptions.metar_availability,assumptions.taf_availability,assumptions.noaa_availability
0,W0002,processed/W0002/data_manifest.json,3bacd4c94b1c267c71024299cb948f95372ad2ddb7332a...,2261775,True,2247194,2247194,1355934,False,"[metar_wind_knots, metar_gust_knots, metar_vis...","[taf_bulletin_age_minutes, taf_lead_minutes, t...",Exclude untimed COR values; retain prior non-C...,Latest bulletin state first; untimed/revised/c...,Retrospective sensitivity branch; source-speci...,observation_time + 5 minutes; sensitivity prox...,"max(body issue time, archive issue time); prox...",observation_time + 5 minutes is retrospective ...


,feature,group,meaning,unit,source,generation_rule,available_at,missing_handling,role
0,airport_id,G0,Stable connection/downstream origin AirportID,category,BTS OriginAirportID,Use stable ID; aliases do not create extra air...,planned before prediction (snapshot assumption),constant missing category fitted in training,predictor
1,route,G0,Downstream origin to destination,category,BTS OriginAirportID/DestAirportID,originID -> destID,planned before prediction (snapshot assumption),constant missing category fitted in training,predictor
2,downstream_airline,G0,Downstream reporting airline,category,BTS Reporting_Airline,Use planned reporting airline identifier,planned before prediction (snapshot assumption),constant missing category fitted in training,predictor
3,downstream_distance,G0,Downstream route distance,miles,BTS Distance,Copy scheduled route distance,planned before prediction,training median plus missing indicator,predictor
4,downstream_crs_elapsed_time,G0,Downstream scheduled elapsed time,minutes,BTS CRSElapsedTime,Copy scheduled duration,planned before prediction,training median plus missing indicator,predictor
...,...,...,...,...,...,...,...,...,...
64,taf_probability_rain_flag,G4B,Probability (PROB) forecast: rain/drizzle indi...,0/1,IEM TAF body / parsed PROB segment,Any active segment reports RA/DZ; NA if active...,Selected bulletin availability proxy=max(body ...,Count is 0 if no same-kind segment is active; ...,predictor
65,taf_probability_snow_flag,G4B,Probability (PROB) forecast: snow/ice/hail-fam...,0/1,IEM TAF body / parsed PROB segment,Any active segment reports SN/SG/PL/GR/GS; NA ...,Selected bulletin availability proxy=max(body ...,Count is 0 if no same-kind segment is active; ...,predictor
66,taf_probability_thunder_flag,G4B,Probability (PROB) forecast: thunderstorm indi...,0/1,IEM TAF body / parsed PROB segment,Any active segment reports TS; NA if active se...,Selected bulletin availability proxy=max(body ...,Count is 0 if no same-kind segment is active; ...,predictor
67,taf_probability_segment_count,G4B,Probability (PROB) forecast: number of active ...,segments,IEM TAF body / parsed PROB segment,Count of same-kind intervals containing the fo...,Selected bulletin availability proxy=max(body ...,Count is 0 if no same-kind segment is active; ...,predictor


## 2. 航班链、飞机映射及机场运行

航班链历史默认使用预测前 24 小时已完成的航段，不重复计入当前上游；其已知到达延误单独作为字段。已知尾号没有观察到有效历史时，计数为零、均值缺失；未知尾号保留缺失。history_window_incomplete 是**已知左截断指示**，不是 flag=0 时的完整覆盖保证。源数据按当地 4 月起飞日期取样，前月起飞而延迟到达/出发的 carry-in 航班可能缺失。

B43 是静态参考清单，以航司＋尾号连接，不能当作逐预测时点的历史机队快照。family 与 variant 单独比较，含截距线性模型不同时冗余展开二者。

机场历史窗口是 \( [t-60\mathrm{min},t) \)，只计已实际出发事件；计划负荷窗口是 \( [t,t+60\mathrm{min}) \)。最终取消状态不能用于事后删去计划负荷。计划快照事前已知是研究假设，仓库没有逐时计划修订版本。

## 3. 天气时点契约

METAR 假设观测后 5 分钟可用，观测年龄最多 120 分钟；这是建模假设，不是源数据提供的接收时间。TAF 先选预测时点已可用的最新公告，再匹配查询时刻 max(下游计划出发, prediction_time)，并处理有效期、修订、取消/NIL 及临时/概率段。NOAA 仅作独立补缺分支，保留质量码、单位核对和逐值来源。缺天气保留样本及缺失标记，覆盖对照须核对评估 ID。

## 4. 实际缺失、映射与覆盖表

以下仅展示本轮确实归档的表。没有对应表不代表该信息源覆盖完整，也不代表该信息组已经启用。

In [5]:
coverage_paths = sorted({p for folder in [RUN_DIR, TABLES] if folder.exists()
                         for p in folder.glob("*.csv")
                         if any(word in p.stem.lower() for word in ["missing", "coverage", "match_rate", "unmatched"])})
for path in coverage_paths:
    display(Markdown(f"**{path.relative_to(RUN_DIR)}**"))
    data = pd.read_csv(path)
    display(Markdown(f"归档表共 {len(data):,} 行；下方展示前 50 行。"))
    display(data.head(50))
if not coverage_paths:
    display(Markdown("本轮没有单独归档缺失/覆盖表；请结合上方审计阅读。"))

**tables\eda_feature_missing.csv**

归档表共 8 行；下方展示前 50 行。

,feature,missing_n,missing_share
0,airport_id,0,0.0
1,route,0,0.0
2,downstream_airline,0,0.0
3,downstream_distance,0,0.0
4,downstream_crs_elapsed_time,0,0.0
5,downstream_day_of_week,0,0.0
6,downstream_sched_hour,0,0.0
7,scheduled_turnaround_minutes,0,0.0


**tables\feature_missing.csv**

归档表共 65 行；下方展示前 50 行。

...

归档表共 4 行；下方展示前 50 行。

,month_utc,metar_available_count,metar_available_sum,metar_available_mean,taf_available_count,taf_available_sum,taf_available_mean,taf_prevailing_available_count,taf_prevailing_available_sum,taf_prevailing_available_mean,taf_latest_state_located_count,taf_latest_state_located_sum,taf_latest_state_located_mean
0,2026-04,554684,552797.0,0.996598,554684,331024.0,0.596779,554684,331024,0.596779,554684,552891,0.996768
1,2026-05,574571,572225.0,0.995917,574571,341795.0,0.594870,574571,341795,0.594870,574571,572530,0.996448
2,2026-06,561972,560174.0,0.996801,561972,335605.0,0.597192,561972,335605,0.597192,561972,559604,0.995786
3,2026-07,570548,561998.0,0.985014,570548,347510.0,0.609081,570548,347510,0.609081,570548,568012,0.995555


**tables\weather_variable_coverage.csv**

归档表共 47 行；下方展示前 50 行。

,feature,numeric_or_flag_present_n,queries,present_share
0,metar_wind_knots,2244576,2261775,0.992396
1,metar_gust_knots,467426,2261775,0.206663
2,metar_visibility_miles,2246828,2261775,0.993391
3,metar_ceiling_ft,1123433,2261775,0.496704
4,metar_precip_amount_in,2143463,2261775,0.947691
5,metar_precip_flag,2247194,2261775,0.993553
6,metar_rain_flag,158281,2261775,0.069981
7,metar_snow_flag,158281,2261775,0.069981
8,metar_thunder_flag,158281,2261775,0.069981
9,metar_age_minutes,2249876,2261775,0.994739
